In [1]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

In [2]:
ds = load_dataset("google/civil_comments")

In [3]:
df = pd.DataFrame(ds['train'])
df

,text,toxicity,severe_toxicity,obscene,threat,insult,identity_attack,sexual_explicit
0,"This is so cool. It's like, 'would you want yo...",0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0
1,Thank you!! This would make my life a lot less...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0
2,This is such an urgent design problem; kudos t...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0
3,Is this something I'll be able to install on m...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0
4,haha you guys are a bunch of losers.,0.893617,0.021277,0.000000,0.0,0.872340,0.021277,0.0
...,...,...,...,...,...,...,...,...
1804869,"Maybe the tax on ""things"" would be collected w...",0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0
1804870,What do you call people who STILL think the di...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0
1804871,"thank you ,,,right or wrong,,, i am following ...",0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0
1804872,Anyone who is quoted as having the following e...,0.621212,0.030303,0.030303,0.0,0.621212,0.045455,0.0


In [4]:
df['label'] = (df['toxicity'] > 0.5).astype(int)

In [5]:
toxic_df = df[df['label'] == 1]
clean_df = df[df['label'] == 0]

In [6]:
sample_size_per_class = 10000
toxic_sampled = toxic_df.sample(n=sample_size_per_class, random_state=42)
clean_sampled = clean_df.sample(n=sample_size_per_class, random_state=42)

In [7]:
balanced_df = pd.concat([toxic_sampled,clean_sampled]).sample(frac=1,random_state=42).reset_index(drop=True)

In [8]:
X_train, X_val, y_train, y_val = train_test_split(balanced_df['text'], balanced_df['label'], test_size=0.2, stratify = balanced_df['label'], random_state=42)

In [9]:
print(f"Training shapes: {X_train.shape}, Validation shapes: {X_val.shape}")

Training shapes: (16000,), Validation shapes: (4000,)


In [10]:
vectorizer = TfidfVectorizer(max_features=10000, stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)

In [11]:
baseline_model = LogisticRegression(max_iter=1000)
baseline_model.fit(X_train_tfidf, y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [12]:
preds = baseline_model.predict(X_val_tfidf)
print("\n--- BASELINE PERFORMANCE REPORT ---")
print(classification_report(y_val, preds))


--- BASELINE PERFORMANCE REPORT ---
              precision    recall  f1-score   support

           0       0.79      0.89      0.84      2000
           1       0.87      0.77      0.82      2000

    accuracy                           0.83      4000
   macro avg       0.83      0.83      0.83      4000
weighted avg       0.83      0.83      0.83      4000



In [13]:
from datasets import Dataset
from transformers import AutoTokenizer

In [14]:
train_dataset = Dataset.from_pandas(pd.DataFrame({'text': X_train, 'label': y_train}))
val_dataset = Dataset.from_pandas(pd.DataFrame({'text': X_val, 'label': y_val}))


In [15]:
model_ckpt = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

In [16]:
def tokenize_text(examples):
    return tokenizer(examples['text'], truncation=True, padding="max_length", max_length=128)#TO HAVE MAX VC LEN OF 128 CUZ SHORT COMMENTS HERE

In [17]:
print("Tokenizing training data...")
tokenized_train = train_dataset.map(tokenize_text, batched=True)

print("Tokenizing validation data...")
tokenized_val = val_dataset.map(tokenize_text, batched=True)

print("\nTokenization complete! Check out the new columns added to your data:")
print(tokenized_train.column_names)

Tokenizing training data...


Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Tokenizing validation data...


Map:   0%|          | 0/4000 [00:00<?, ? examples/s]


Tokenization complete! Check out the new columns added to your data:
['text', 'label', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask']


In [18]:
import numpy as np
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


In [19]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_ckpt, 
    num_labels=2
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [20]:
#we use logits instead of probabilities for the loss function, so we need to use argmax to get the predicted class
#also logits take less computational resources than probabilities, so we use logits for efficiency
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [21]:
batch_size = 32

training_args = TrainingArguments(
    output_dir="./distilbert-toxicity-model",
    num_train_epochs=3,              # 3 epochs is usually the sweet spot for fine-tuning
    learning_rate=2e-5,              # Standard fine-tuning LR
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    weight_decay=0.01,               # Prevents overfitting
    eval_strategy="epoch",   # Evaluate at the end of every epoch
    save_strategy="epoch",           # Save a checkpoint at the end of every epoch
    load_best_model_at_end=True,     # Keep the best performing model
    fp16=True,                       # CRITICAL: Cuts memory usage in half, speeds up training
    logging_steps=50,                # Print updates frequently
)

In [22]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,   # From your previous tokenization step
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

In [23]:
trainer.train()

# 6. Save the final model and tokenizer for Day 4 & Day 5
trainer.save_model("./best_toxicity_model")
tokenizer.save_pretrained("./best_toxicity_model")
print("Training complete and model saved locally!")

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.310259,0.273790,0.891750,0.891703,0.892426,0.891750
2,0.211137,0.275013,0.889750,0.889735,0.889965,0.889750
3,0.145401,0.307682,0.892250,0.892232,0.892505,0.892250


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete and model saved locally!


In [24]:
model_path = "./best_toxicity_model"
loaded_model = AutoModelForSequenceClassification.from_pretrained(model_path)
inference_trainer = Trainer(model=loaded_model)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [25]:
print("Running predictions on the validation set")
predictions_output = inference_trainer.predict(tokenized_val)

Running predictions on the validation set


In [26]:
raw_logits = predictions_output.predictions
y_preds = np.argmax(raw_logits, axis=-1)
y_true = predictions_output.label_ids


In [27]:
error_df = pd.DataFrame({
    'text': X_val.values,
    'actual_label': y_true,
    'predicted_label': y_preds
})

In [28]:
false_positives = error_df[(error_df['actual_label'] == 0) & (error_df['predicted_label'] == 1)]
false_negatives = error_df[(error_df['actual_label'] == 1) & (error_df['predicted_label'] == 0)]


In [30]:
print(f"\nTotal False Positives (Unfairly Censored): {len(false_positives)}")
print(f"Total False Negatives (Missed Toxicity): {len(false_negatives)}\n")

print(f"\nTotal False Positives (Unfairly Censored): {len(false_positives)}")
print(f"Total False Negatives (Missed Toxicity): {len(false_negatives)}\n")

print(" TOP FALSE POSITIVES (Model thought these were toxic)")

pd.set_option('display.max_colwidth', None)
print(false_positives['text'].sample(min(5, len(false_positives)), random_state=42).to_string(index=False))

print("\nTOP FALSE NEGATIVES (Model missed this toxicity)")
print(false_negatives['text'].sample(min(5, len(false_negatives)), random_state=42).to_string(index=False))


Total False Positives (Unfairly Censored): 258
Total False Negatives (Missed Toxicity): 175


Total False Positives (Unfairly Censored): 258
Total False Negatives (Missed Toxicity): 175

 TOP FALSE POSITIVES (Model thought these were toxic)
....Except that's not how it works.  This isn't about "how many Americans are killed by terrorists."  it goes deeper than that-- it's a values argument.  Muslims are overwhelmingly un-American and do not share the same notions of liberty, freedom, and democracy as we do.  Look at Pew Research polls of the Muslim world:  The overwhelming majority of them believe Sharia Law is the ONLY law.  The majority believe there is only ONE interpretation of Sharia Law, and that it also supersedes any American government writ or law.  The majority of Muslims directly support, indirectly support, or at the very last sympathize with the following people: Islamists, Jihadists, Islamist extremists, Conservative sects of Islam, or variations therein.\n.\nThe Muslim 

In [31]:
import shap
import transformers
import scipy as sp


In [32]:
classifier_pipeline = transformers.pipeline(
    "text-classification", 
    model="./best_toxicity_model", 
    tokenizer="./best_toxicity_model",
    return_all_scores=True # SHAP needs probabilities for all classes
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [33]:
explainer = shap.Explainer(classifier_pipeline)
text_to_explain = [
    "Muslims are overwhelmingly un-American and do not share the same notions of liberty. Look at polls of the Muslim world: The majority believe Sharia Law is the ONLY law."
]
shap_values = explainer(text_to_explain)


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [34]:
shap.plots.text(shap_values)